### Five SRL behaviors (L & I embodied)

Prompts live in **`all-prompts/L_and_I_Prompt_<BEHAVIOR>.txt`** (same `[***NEW_MESSAGE***]` few-shot layout). Use **`prepare_embodied_csv_for_action(df, action, student_name)`** so the CSV matches what each prompt asserts.

| Behavior | Modalities kept (after shaping) | `Speaker:` line |
|----------|-----------------------------------|-----------------|
| **ENACTING** | gaze (Screen only), movement, action, state | No |
| **MONITORING** | same as ENACTING | No |
| **INTERACTING** | gaze, speech, state | Yes |
| **REFLECTING** | gaze, speech, gesture | Yes |
| **PLANNING** | gaze, speech, gesture | Yes |

- **Few-shot builders:** `enacting_fewshot_messages`, `monitoring_fewshot_messages`, `interacting_fewshot_messages`, `reflecting_fewshot_messages`, `planning_fewshot_messages` → each returns the prefix; append one **`{"role":"user","content": api_csv_string}`** turn.

- **Notebook layout:** **§1b Pipeline inputs** (one place to swap CSVs) → **§2 API** → **§3 ENACTING** → **§4 INTERACTING** → **§5 MONITORING** → **§6 PLANNING** → **§7 REFLECTING** → **§9 Inference pipeline** onward.




## 1. Shared — paths, prompts & CSV helper

`PROJECT_ROOT`, `PROMPTS_DIR`, `SPLIT_STRING`, `STUDENT_DAY_CSV_DIR`, `load_srl_prompt_parts`, `srl_fewshot_messages`, and `prepare_embodied_csv_for_action` — run this before any few-shot cells for the five behaviors.


In [15]:
from pathlib import Path
import re

PROJECT_ROOT = Path("/Users/caiwansun/Desktop/LLM_L_AND_I_PROJECT")

# Shared delimiter in all `all-prompts/L_and_I_Prompt_*.txt` files
SPLIT_STRING = "\n[***NEW_MESSAGE***]\n"

STUDENT_DAY_CSV_DIR = PROJECT_ROOT / "study-data-per-student-day-behavior"
PROMPTS_DIR = PROJECT_ROOT / "all-prompts"


def load_srl_prompt_parts(behavior: str) -> list:
    """Load [system, demo user, demo assistant] parts for L_and_I_Prompt_<BEHAVIOR>.txt."""
    name = (behavior or "").strip().upper()
    path = PROMPTS_DIR / f"L_and_I_Prompt_{name}.txt"
    raw = path.read_text(encoding="utf-8")

    # Normalize mixed newlines in source prompt files (CRLF/CR -> LF)
    # so SPLIT_STRING parsing is stable across editors.
    raw = raw.replace("\r\n", "\n").replace("\r", "\n")

    parts = raw.split(SPLIT_STRING)
    if len(parts) != 3:
        raise ValueError(f"{path.name}: expected 3 blocks split by SPLIT_STRING; got {len(parts)}")

    return parts


def srl_fewshot_messages(model_id: str, prompt_parts: list) -> list:
    """OpenAI: system + user + assistant. Claude: user + user + assistant."""
    mid = (model_id or "").lower()
    claude = "claude" in mid or "anthropic" in mid
    if claude:
        return [
            {"role": "user", "content": prompt_parts[0].strip()},
            {"role": "user", "content": prompt_parts[1].strip()},
            {"role": "assistant", "content": prompt_parts[2].strip()},
        ]
    return [
        {"role": "system", "content": prompt_parts[0].strip()},
        {"role": "user", "content": prompt_parts[1].strip()},
        {"role": "assistant", "content": prompt_parts[2].strip()},
    ]


def prepare_embodied_csv_for_action(df, action: str, student_name: str):
    """Normalize a modality CSV for the API.

    enacting | monitoring: drop gesture & speech; keep gaze rows only when data == "Screen" (no Speaker line).
    interacting: drop movement, action, gesture; prefix with Speaker: <student_name>.
    reflecting | planning: drop movement, action, state; prefix with Speaker: <student_name>.
    """
    import pandas as pd

    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame")
    act = (action or "").strip().lower()
    if act not in {"enacting", "monitoring", "interacting", "reflecting", "planning"}:
        raise ValueError(
            f"Unknown action {action!r}; use enacting|monitoring|interacting|reflecting|planning"
        )

    d = df.copy().fillna("")

    # Keep only the columns used by prompts / downstream validators.
    # This matches the compact CSV format used in the working notebook.
    core_cols = [c for c in ["start_time", "end_time", "modality", "data"] if c in d.columns]
    missing = [c for c in ["start_time", "end_time", "modality", "data"] if c not in d.columns]
    if missing:
        raise ValueError(f"Input CSV missing required columns: {missing}")
    d = d[core_cols].copy()

    d["data"] = d["data"].replace("not moving", "stationary")

    if act in {"enacting", "monitoring"}:
        d = d[~d["modality"].isin(["gesture", "speech"])]
        d = d[(d["modality"] != "gaze") | (d["data"] == "Screen")]
    elif act == "interacting":
        d = d[~d["modality"].isin(["movement", "action", "gesture"])]
    elif act in {"reflecting", "planning"}:
        d = d[~d["modality"].isin(["movement", "action", "state"])]

    body = d.to_csv(index=False)
    if act in {"interacting", "reflecting", "planning"}:
        body = f"Speaker: {student_name}\n\n" + body
    return body, d


def _project_path(maybe_path) -> Path:
    p = Path(maybe_path)
    return p if p.is_absolute() else (PROJECT_ROOT / p)


def infer_student_name_from_filename(stem: str) -> str:
    m = re.search(r"Student\s+(.+?)\s+-\s+day", stem, flags=re.IGNORECASE)
    return m.group(1).strip() if m else ""


def branch_slug_from_stem(stem: str) -> str:
    s = stem.strip()
    s = re.sub(r"(?i)^L&I\s*-\s*embodied\s*-\s*", "", s)
    s = re.sub(r"(?i)\s*-\s*interacting\s*$", "", s)
    s = re.sub(r"(?i)\s*-\s*enacting\s*$", "", s)
    s = re.sub(r"(?i)\s*-\s*monitoring\s*$", "", s)
    s = re.sub(r"(?i)\s*-\s*planning\s*$", "", s)
    s = re.sub(r"(?i)\s*-\s*reflecting\s*$", "", s)
    s = re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")
    return s or "RUN"


def build_embodied_branch_pack(action: str, csv_path, student_name: str | None = None, df=None):
    """Load + normalize one embodied-behavior CSV, and derive stable naming metadata.

    Returns a dict used by each branch cell + the optional multi-model runners.
    """
    import pandas as pd

    act = (action or "").strip().lower()
    if act not in {"enacting", "monitoring", "interacting", "reflecting", "planning"}:
        raise ValueError(f"Unknown action {action!r}")

    csv_path = _project_path(csv_path)
    stem = csv_path.stem

    inferred = infer_student_name_from_filename(stem)
    student = (student_name if student_name is not None else inferred)

    if act in {"enacting", "monitoring"}:
        student_for_prepare = ""
    else:
        if not isinstance(student, str) or not student.strip():
            raise ValueError(
                f"{csv_path.name}: could not infer student name from filename; "
                f"pass student_name=... to build_embodied_branch_pack(...)"
            )
        student_for_prepare = student.strip()

    if df is None:
        d0 = pd.read_csv(csv_path).fillna("")
    else:
        d0 = df.copy().fillna("")

    api_csv, d_model = prepare_embodied_csv_for_action(d0, act, student_for_prepare)

    slug = branch_slug_from_stem(stem)
    behavior_label = act.upper()
    output_stem = f"{behavior_label}_{slug}"
    processor_label = f"{behavior_label} {slug.replace('_', ' ')}"

    demo_model_id = globals().get("DEMO_MODEL_ID") or globals().get("PREVIEW_MODEL_ID") or "gpt-5.2"

    return {
        "action": act,
        "csv_path": csv_path,
        "student_name": student_for_prepare if act in {"enacting", "monitoring"} else student_for_prepare,
        "inferred_student": inferred,
        "api_csv": api_csv,
        "df_model": d_model,
        "behavior_label": behavior_label,
        "output_stem": output_stem,
        "processor_label": processor_label,
        "demo_model_id": demo_model_id,
    }


## 1b. Pipeline inputs (edit per student/day)

Put **one CSV path per behavior** here. Everything downstream (`messages*`, output stems, default segment labels) is derived automatically.

- Paths can be **absolute** or **relative to `PROJECT_ROOT`**.
- For **INTERACTING / PLANNING / REFLECTING**, filenames should include `Student <Name> - day …` so `Speaker:` can be inferred.


In [16]:
from pathlib import Path

# Used only to decide OpenAI vs Claude few-shot role layout.
DEMO_MODEL_ID = "gpt-5.2"

# Optional: keep legacy notebooks happy.
PREVIEW_MODEL_ID = DEMO_MODEL_ID

PIPELINE_INPUTS = {
    "demo_model_id": DEMO_MODEL_ID,
    "enacting": "L&I - Student Mastersheet - EN-MESSI-D2.csv",
    "interacting": Path("study-data-per-student-day-behavior") / "L&I - embodied - Student Rose - day 1 - interacting.csv",
    "monitoring": Path("study-data-per-student-day-behavior") / "L&I - embodied - Student DaPaw - day 1 - monitoring.csv",
    "planning": Path("study-data-per-student-day-behavior") / "L&I - embodied - Student DaPaw - day 1 - planning.csv",
    "reflecting": Path("study-data-per-student-day-behavior") / "L&I - embodied - Student DaPaw - day 1 - reflecting.csv",
    # Keep one CSV per core behavior; optional companion runs removed.
}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEMO_MODEL_ID:", DEMO_MODEL_ID)
for k in ("enacting", "interacting", "monitoring", "planning", "reflecting"):
    p = _project_path(PIPELINE_INPUTS[k])
    print(f"PIPELINE_INPUTS[{k!r}] ->", p, "exists:", p.exists())


PROJECT_ROOT: /Users/caiwansun/Desktop/LLM_L_AND_I_PROJECT
DEMO_MODEL_ID: gpt-5.2
PIPELINE_INPUTS['enacting'] -> /Users/caiwansun/Desktop/LLM_L_AND_I_PROJECT/L&I - Student Mastersheet - EN-MESSI-D2.csv exists: True
PIPELINE_INPUTS['interacting'] -> /Users/caiwansun/Desktop/LLM_L_AND_I_PROJECT/study-data-per-student-day-behavior/L&I - embodied - Student Rose - day 1 - interacting.csv exists: True
PIPELINE_INPUTS['monitoring'] -> /Users/caiwansun/Desktop/LLM_L_AND_I_PROJECT/study-data-per-student-day-behavior/L&I - embodied - Student DaPaw - day 1 - monitoring.csv exists: True
PIPELINE_INPUTS['planning'] -> /Users/caiwansun/Desktop/LLM_L_AND_I_PROJECT/study-data-per-student-day-behavior/L&I - embodied - Student DaPaw - day 1 - planning.csv exists: True
PIPELINE_INPUTS['reflecting'] -> /Users/caiwansun/Desktop/LLM_L_AND_I_PROJECT/study-data-per-student-day-behavior/L&I - embodied - Student DaPaw - day 1 - reflecting.csv exists: True


## 2. Shared — API token & headers

Run **Configuration** before `get_available_models` / `run_pipeline`.

Few-shot role layout (OpenAI vs Claude) is controlled by **`DEMO_MODEL_ID`** in **Pipeline inputs** (§1b). You can still export `PREVIEW_MODEL_ID` if you want, but it is optional.


In [17]:
## Configuration
import os
import requests

BASE_URL = "https://prod-api.vanderbilt.ai"
AMP_TOKEN = "amp-ba734aaf-cc64-4a4f-9fba-21322460c34c"

if AMP_TOKEN.strip():
    os.environ["AMP_TOKEN"] = AMP_TOKEN.strip()
AMP_TOKEN = os.environ.get("AMP_TOKEN", "").strip()
if not AMP_TOKEN:
    raise RuntimeError(
        "Set AMP_TOKEN: paste into AMP_TOKEN = '' in this cell, or export AMP_TOKEN='…', then re-run."
    )

headers = {
    "Authorization": f"Bearer {AMP_TOKEN}",
    "Content-Type": "application/json",
}

## 3. ENACTING branch

Few-shot → mastersheet import → default **`messages`** (CSV path comes from **§1b Pipeline inputs**).


In [18]:
# ENACTING: few-shot from all-prompts (use with `messages` + MESSI CSV below).
prompt_split_enacting = load_srl_prompt_parts("ENACTING")


def enacting_fewshot_messages(model_id: str) -> list:
    return srl_fewshot_messages(model_id, prompt_split_enacting)



In [19]:
import pandas as pd

# ENACTING: load the CSV from PIPELINE_INPUTS, normalize, build default `messages`.
BRANCH_ENACTING = build_embodied_branch_pack("enacting", PIPELINE_INPUTS["enacting"])

DATA_PATH = BRANCH_ENACTING["csv_path"]
df = pd.read_csv(DATA_PATH).fillna("")

data_csv_string = BRANCH_ENACTING["api_csv"]
df_enacting_model = BRANCH_ENACTING["df_model"]

DEFAULT_BEHAVIOR_LABEL = BRANCH_ENACTING["behavior_label"]
OUTPUT_STEM_PREFIX = BRANCH_ENACTING["output_stem"]
AMALIA_PROCESSOR_LABEL = BRANCH_ENACTING["processor_label"]

messages = enacting_fewshot_messages(BRANCH_ENACTING["demo_model_id"]) + [
    {"role": "user", "content": data_csv_string},
]

print("ENACTING csv:", str(DATA_PATH))
print("ENACTING output_stem:", OUTPUT_STEM_PREFIX)
print("ENACTING message count:", len(messages))
print("ENACTING prompt chars:", [len(str(m.get("content", ""))) for m in messages[:3]])
print("ENACTING csv chars:", len(str(messages[-1].get("content", ""))))
df_enacting_model.head()


ENACTING csv: /Users/caiwansun/Desktop/LLM_L_AND_I_PROJECT/L&I - Student Mastersheet - EN-MESSI-D2.csv
ENACTING output_stem: ENACTING_L_I_Student_Mastersheet_EN_MESSI_D2
ENACTING message count: 4
ENACTING prompt chars: [4397, 927, 6292]
ENACTING csv chars: 6067


,start_time,end_time,modality,data
0,0:00:00,,state,waterthinking
1,0:00:00,0:00:05,movement,stationary
3,0:00:06,0:00:09,movement,moving
4,0:00:07,0:04:38,gaze,Screen
5,0:00:08,0:00:14,movement,stationary


## 4. INTERACTING branch

Few-shot → per-student discourse CSV → **`messages_interacting`** (CSV path comes from **§1b Pipeline inputs**).


In [20]:
# INTERACTING: few-shot from all-prompts (use with `messages_interacting` + discourse CSV below).
prompt_split_interacting = load_srl_prompt_parts("INTERACTING")


def interacting_fewshot_messages(model_id: str) -> list:
    return srl_fewshot_messages(model_id, prompt_split_interacting)

In [21]:
# INTERACTING: load the CSV from PIPELINE_INPUTS, normalize, build `messages_interacting`.
# Few-shot prefix is defined above (**INTERACTING — few-shot from Prompt_INTERACTING.txt**); run that cell first.
BRANCH_INTERACTING = build_embodied_branch_pack("interacting", PIPELINE_INPUTS["interacting"])

api_csv_interacting = BRANCH_INTERACTING["api_csv"]
_df_inter_model = BRANCH_INTERACTING["df_model"]

DEFAULT_BEHAVIOR_LABEL_INTER = BRANCH_INTERACTING["behavior_label"]
OUTPUT_STEM_PREFIX_INTER = BRANCH_INTERACTING["output_stem"]
AMALIA_PROCESSOR_LABEL_INTER = BRANCH_INTERACTING["processor_label"]

messages_interacting = interacting_fewshot_messages(BRANCH_INTERACTING["demo_model_id"]) + [
    {"role": "user", "content": api_csv_interacting},
]

print("INTERACTING csv:", str(BRANCH_INTERACTING["csv_path"]))
print("INTERACTING output_stem:", OUTPUT_STEM_PREFIX_INTER)
print("INTERACTING message count:", len(messages_interacting))
_df_inter_model.head()


INTERACTING csv: /Users/caiwansun/Desktop/LLM_L_AND_I_PROJECT/study-data-per-student-day-behavior/L&I - embodied - Student Rose - day 1 - interacting.csv
INTERACTING output_stem: INTERACTING_Student_Rose_day_1
INTERACTING message count: 4


,start_time,end_time,modality,data
0,0:00:01,,speech,Teacher - Ms Hughes: Okay.
1,0:00:02,,speech,Teacher - Ms Hughes: So I want you to take a l...
2,0:00:05,0:00:10,gaze,Screen
3,0:00:05,,speech,Teacher - Ms Hughes: So what is Rose starting ...
4,0:00:08,,speech,Student - Rose: Air.


## 5. MONITORING branch

Few-shot → per-student day CSV → **`messages_monitoring`** (CSV path comes from **§1b Pipeline inputs**).


In [22]:
# MONITORING: few-shot from all-prompts (use with `messages_monitoring` + per-student day CSV below).
import pandas as pd

prompt_split_monitoring = load_srl_prompt_parts("MONITORING")


def monitoring_fewshot_messages(model_id: str) -> list:
    return srl_fewshot_messages(model_id, prompt_split_monitoring)


BRANCH_MONITORING = build_embodied_branch_pack("monitoring", PIPELINE_INPUTS["monitoring"])

api_csv_mon = BRANCH_MONITORING["api_csv"]
_df_mon_model = BRANCH_MONITORING["df_model"]

messages_monitoring = monitoring_fewshot_messages(BRANCH_MONITORING["demo_model_id"]) + [
    {"role": "user", "content": api_csv_mon},
]

print("MONITORING csv:", str(BRANCH_MONITORING["csv_path"]))
print("MONITORING output_stem:", BRANCH_MONITORING["output_stem"])
print("MONITORING message count:", len(messages_monitoring))
_df_mon_model.head()


MONITORING csv: /Users/caiwansun/Desktop/LLM_L_AND_I_PROJECT/study-data-per-student-day-behavior/L&I - embodied - Student DaPaw - day 1 - monitoring.csv
MONITORING output_stem: MONITORING_Student_DaPaw_day_1
MONITORING message count: 4


,start_time,end_time,modality,data
0,0:00:05,0:00:10,gaze,Screen
1,0:00:10,0:00:15,gaze,Screen
2,0:00:15,0:00:20,gaze,Screen
3,0:00:20,0:00:25,gaze,Screen
4,0:00:23,0:00:30,movement,stationary


## 6. PLANNING branch

Few-shot → per-student day CSV → **`messages_planning`** (CSV path comes from **§1b Pipeline inputs**).


In [23]:
# PLANNING: few-shot from all-prompts (use with `messages_planning` + per-student day CSV below).
import pandas as pd

prompt_split_planning = load_srl_prompt_parts("PLANNING")


def planning_fewshot_messages(model_id: str) -> list:
    return srl_fewshot_messages(model_id, prompt_split_planning)


BRANCH_PLANNING = build_embodied_branch_pack("planning", PIPELINE_INPUTS["planning"])

api_csv_plan = BRANCH_PLANNING["api_csv"]
_df_plan_model = BRANCH_PLANNING["df_model"]

messages_planning = planning_fewshot_messages(BRANCH_PLANNING["demo_model_id"]) + [
    {"role": "user", "content": api_csv_plan},
]

print("PLANNING csv:", str(BRANCH_PLANNING["csv_path"]))
print("PLANNING output_stem:", BRANCH_PLANNING["output_stem"])
print("PLANNING message count:", len(messages_planning))
_df_plan_model.head()


PLANNING csv: /Users/caiwansun/Desktop/LLM_L_AND_I_PROJECT/study-data-per-student-day-behavior/L&I - embodied - Student DaPaw - day 1 - planning.csv
PLANNING output_stem: PLANNING_Student_DaPaw_day_1
PLANNING message count: 4


,start_time,end_time,modality,data
0,0:00:01,,speech,Teacher - Ms Hughes: Okay.
1,0:00:02,,speech,Teacher - Ms Hughes: So I want you to take a l...
2,0:00:05,0:00:10,gaze,Screen
3,0:00:05,,speech,Teacher - Ms Hughes: So what is Rose starting ...
4,0:00:08,,speech,Student - Rose: Air.


## 7. REFLECTING branch

Few-shot → per-student day CSV → **`messages_reflecting`** (CSV path comes from **§1b Pipeline inputs**).


In [24]:
# REFLECTING: few-shot from all-prompts (use with `messages_reflecting` + per-student day CSV below).
import pandas as pd

prompt_split_reflecting = load_srl_prompt_parts("REFLECTING")


def reflecting_fewshot_messages(model_id: str) -> list:
    return srl_fewshot_messages(model_id, prompt_split_reflecting)


BRANCH_REFLECTING = build_embodied_branch_pack("reflecting", PIPELINE_INPUTS["reflecting"])

api_csv_refl = BRANCH_REFLECTING["api_csv"]
_df_refl_model = BRANCH_REFLECTING["df_model"]

messages_reflecting = reflecting_fewshot_messages(BRANCH_REFLECTING["demo_model_id"]) + [
    {"role": "user", "content": api_csv_refl},
]

print("REFLECTING csv:", str(BRANCH_REFLECTING["csv_path"]))
print("REFLECTING output_stem:", BRANCH_REFLECTING["output_stem"])
print("REFLECTING message count:", len(messages_reflecting))
_df_refl_model.head()


REFLECTING csv: /Users/caiwansun/Desktop/LLM_L_AND_I_PROJECT/study-data-per-student-day-behavior/L&I - embodied - Student DaPaw - day 1 - reflecting.csv
REFLECTING output_stem: REFLECTING_Student_DaPaw_day_1
REFLECTING message count: 4


,start_time,end_time,modality,data
0,0:00:01,,speech,Teacher - Ms Hughes: Okay.
1,0:00:02,,speech,Teacher - Ms Hughes: So I want you to take a l...
2,0:00:05,0:00:10,gaze,Screen
3,0:00:05,,speech,Teacher - Ms Hughes: So what is Rose starting ...
4,0:00:08,,speech,Student - Rose: Air.


In [ ]:
# reserved


## 8. Inference pipeline

Run **Unified Inference Pipeline** once (defines the base `run_pipeline` and sets `_run_pipeline_original`). Then run **Hotfix** (thin wrapper: JSON-retry + always looks up the latest `_run_pipeline_original`). Re-run **Pipeline** after any edit to `run_pipeline`, then **Hotfix** again. Optionally run the **print** cell to confirm definitions.


In [25]:
# ===== Unified Inference Pipeline =====
import os
import re
import json
import time
import requests
import pandas as pd


def get_available_models(base_url: str, headers: dict):
    url = f"{base_url}/available_models"
    r = requests.get(url, headers=headers, timeout=30)
    r.raise_for_status()
    payload = r.json()

    data = payload.get("data", {}) if isinstance(payload, dict) else {}
    models = data.get("models", []) if isinstance(data, dict) else []
    default_model = data.get("default", {}) if isinstance(data, dict) else {}

    model_ids = [m.get("id") for m in models if isinstance(m, dict) and isinstance(m.get("id"), str)]
    model_ids = list(dict.fromkeys(model_ids))
    default_id = default_model.get("id") if isinstance(default_model, dict) else None

    model_lookup = {
        m.get("id"): m for m in models
        if isinstance(m, dict) and isinstance(m.get("id"), str)
    }
    return model_ids, default_id, payload, model_lookup


def _unwrap_code_fence(text: str) -> str:
    s = text.strip()
    m = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", s, re.IGNORECASE)
    return m.group(1).strip() if m else s


def _parse_time_to_seconds(time_str: str):
    if not isinstance(time_str, str):
        return None
    s = time_str.strip()
    if not s:
        return None

    parts = s.split(":")
    try:
        if len(parts) == 3:
            h, m, sec = parts
            return int(h) * 3600 + int(m) * 60 + float(sec)
        if len(parts) == 2:
            m, sec = parts
            return int(m) * 60 + float(sec)
    except ValueError:
        return None
    return None


def _seconds_to_hhmmss(seconds: float) -> str:
    total = max(0, int(round(seconds)))
    h = total // 3600
    m = (total % 3600) // 60
    s = total % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def _normalize_time_text(time_str: str) -> str | None:
    sec = _parse_time_to_seconds(time_str)
    if sec is None:
        return None
    return _seconds_to_hhmmss(sec)


def parse_model_json_strict(text):
    """ENACTING default: optional ``` fence strip, then json.loads only (no raw_decode / regex salvage)."""
    if isinstance(text, (dict, list)):
        return text
    if not isinstance(text, str):
        return None
    s = _unwrap_code_fence(text).strip()
    try:
        return json.loads(s)
    except Exception:
        return None


def safe_parse_json(text):
    # Already-structured payload from provider.
    if isinstance(text, (dict, list)):
        return text

    if not isinstance(text, str):
        return None

    s = _unwrap_code_fence(text).strip()

    # 1) Exact JSON.
    try:
        return json.loads(s)
    except Exception:
        pass

    # 2) First valid JSON value (allows trailing text).
    try:
        decoder = json.JSONDecoder()
        obj, _ = decoder.raw_decode(s)
        return obj
    except Exception:
        pass

    # 3) Fallback object/array span parse.
    for pat in (r"\{[\s\S]*\}", r"\[[\s\S]*\]"):
        m = re.search(pat, s)
        if not m:
            continue
        cand = m.group(0).strip()
        try:
            return json.loads(cand)
        except Exception:
            try:
                obj, _ = json.JSONDecoder().raw_decode(cand)
                return obj
            except Exception:
                pass

    return None


def normalize_output_data(data):
    """Normalize provider payload into (parsed_json_or_none, error_message_or_none, raw_text)."""
    if isinstance(data, dict):
        # Backend may wrap error in structured dict/list; coerce safely to text.
        err = data.get("error") or data.get("message")
        if err is not None:
            err_text = str(err).strip()
            if err_text:
                return None, err_text, json.dumps(data, ensure_ascii=False)
        return data, None, json.dumps(data, ensure_ascii=False)

    if isinstance(data, list):
        if len(data) == 0:
            return None, "Empty list response", "[]"
        return data, None, json.dumps(data, ensure_ascii=False)

    if isinstance(data, str):
        s = data.strip()
        if not s:
            return None, "Empty string response", ""
        if s.lower().startswith("error:"):
            return None, s, s
        parsed = safe_parse_json(s)
        return parsed, None if parsed is not None else None, s

    return None, f"Unsupported response type: {type(data).__name__}", str(data)



def _extract_screen_gaze_windows_from_csv(csv_text: str):
    df_local = pd.read_csv(pd.io.common.StringIO(csv_text)).fillna("")
    df_local["modality"] = df_local["modality"].astype(str)
    df_local["data"] = df_local["data"].astype(str)

    gaze = df_local[(df_local["modality"] == "gaze") & (df_local["data"] == "Screen")]
    windows = []
    for _, r in gaze.iterrows():
        s = _parse_time_to_seconds(str(r.get("start_time", "")))
        e = _parse_time_to_seconds(str(r.get("end_time", "")))
        if s is None or e is None:
            continue
        windows.append((s, e))
    return windows


def normalize_and_validate_segments(segments: list, csv_text: str, default_label: str = "ENACTING"):
    # Keep segments when times parse. No ordering rule (does not require time_in < time_out). Dedup + sort only.
    if not isinstance(segments, list):
        return []

    out = []
    seen = set()

    for seg in segments:
        if not isinstance(seg, dict):
            continue

        s = _parse_time_to_seconds(str(seg.get("time_in", "")))
        e = _parse_time_to_seconds(str(seg.get("time_out", "")))
        if s is None or e is None:
            continue

        row = {
            "justification": str(seg.get("justification", "")).strip(),
            "time_in": _seconds_to_hhmmss(s),
            "time_out": _seconds_to_hhmmss(e),
            "label": str(seg.get("label", default_label) or default_label),
        }

        # Exact duplicate suppression only.
        dedup_key = (row["justification"], row["time_in"], row["time_out"], row["label"])
        if dedup_key in seen:
            continue
        seen.add(dedup_key)

        out.append(row)

    out.sort(key=lambda x: (
        _parse_time_to_seconds(x.get("time_in", "")),
        _parse_time_to_seconds(x.get("time_out", "")),
    ))
    return out


def compact_segment_text(segments: list, max_words: int = 12, max_chars: int = 90, default_label: str = "ENACTING"):
    """Optional post-process: shorten each justification (used only when run_pipeline(..., compact_justifications=True))."""
    if not isinstance(segments, list):
        return []

    compacted = []
    for seg in segments:
        if not isinstance(seg, dict):
            continue

        j = str(seg.get("justification", "")).strip()
        if not j:
            j = "Verified event"

        # Keep only the first sentence-like chunk.
        j = re.split(r"(?<=[.!?])\s+", j)[0].strip()

        # Enforce short text budget.
        words = j.split()
        if len(words) > max_words:
            j = " ".join(words[:max_words])
        if len(j) > max_chars:
            j = j[:max_chars].rstrip()

        compacted.append({
            "justification": j,
            "time_in": seg.get("time_in", ""),
            "time_out": seg.get("time_out", ""),
            "label": seg.get("label", default_label),
        })

    return compacted


def sanitize_filename(text: str) -> str:
    return re.sub(r"[^a-zA-Z0-9._-]", "_", text)


# Single set of defaults for every model_id (fair cross-model comparison).
PIPELINE_DEFAULT_MAX_ATTEMPTS = 6
PIPELINE_TRANSIENT_HTTP_SLEEP_CAP_S = 180
PIPELINE_TRANSIENT_HTTP_SLEEP_MULT_S = 20


def run_pipeline(model_id: str, messages: list, base_url: str, headers: dict,
                 output_dir: str = ".", output_stem: str | None = None,
                 max_tokens: int = 4000, temperature: float = 0,
                 max_attempts: int = PIPELINE_DEFAULT_MAX_ATTEMPTS, strict_mode: bool = True,
                 segments_only: bool = True,
                 request_timeout: float = 600,
                 model_object: dict | None = None,
                 parse_json_lenient: bool = False,
                 default_segment_label: str = "ENACTING",
                 compact_justifications: bool = False):
    # Five SRL behaviors: strict JSON only (parse_json_lenient=False) — no safe_parse_json salvage.

    def prepare_messages_for_model(_model_id_inner: str, msg_list: list):
        # Claude: first block must be user, not system (instructions → user; demo CSV already user).
        mid = (_model_id_inner or "").lower()
        if not ("claude" in mid or "anthropic" in mid):
            return msg_list
        out = []
        for i, item in enumerate(msg_list):
            if not isinstance(item, dict):
                out.append(item)
                continue
            role = str(item.get("role", "user"))
            if i == 0 and role == "system":
                out.append({**item, "role": "user"})
            else:
                out.append(item)
        return out

    def coerce_messages_to_text(msg_list: list):
        out = []
        for item in msg_list:
            role = str(item.get("role", "user"))
            content = item.get("content", "")
            if isinstance(content, str):
                text = content
            elif isinstance(content, (dict, list)):
                text = json.dumps(content, ensure_ascii=False)
            else:
                text = str(content)
            out.append({"role": role, "content": text})
        return out

    model_messages = coerce_messages_to_text(prepare_messages_for_model(model_id, messages))

    csv_text = ""
    for m in reversed(model_messages):
        if m.get("role") == "user" and "start_time" in str(m.get("content", "")):
            csv_text = str(m.get("content", ""))
            break

    tag = sanitize_filename(model_id)
    _lbl = sanitize_filename((default_segment_label or "RUN").strip())
    stem = (output_stem or f"{_lbl}_{tag}").strip()
    json_path = os.path.join(output_dir, f"{stem}.json")

    # Use caller-provided token budget consistently across models.
    cap = max(1, int(max_tokens))
    n_sched = max(4, int(max_attempts))
    _out_tok_cap = 128000  # lower if the gateway rejects large max_tokens
    _mults = (1, 2, 3, 4)
    token_schedule = [
        min(cap * _mults[min(i, len(_mults) - 1)], _out_tok_cap)
        for i in range(n_sched)
    ]

    # Same request shape for every provider (model must be an object on this gateway).
    if isinstance(model_object, dict) and model_object:
        _model_field = dict(model_object)
    else:
        _model_field = {"id": model_id}

    payload = {
        "data": {
            "temperature": temperature,
            "max_tokens": token_schedule[0],
            "dataSources": [],
            "messages": model_messages,
            "response_format": {"type": "json_object"},
            "options": {
                "skipRag": True,
                "ragOnly": False,
                "model": _model_field,
            },
        }
    }

    last_error = None
    for attempt in range(1, max_attempts + 1):
        token_idx = min(attempt - 1, len(token_schedule) - 1)
        payload["data"]["max_tokens"] = token_schedule[token_idx]

        if attempt == 1:
            _om = payload["data"]["options"].get("model")
            print(
                "[run_pipeline] options.model",
                type(_om).__name__,
                _om if isinstance(_om, str) else (list(_om.keys()) if isinstance(_om, dict) else _om),
            )

        chat_url = f"{base_url.rstrip('/')}/chat"
        _req_to = float(request_timeout)
        print(
            f"[run_pipeline][attempt {attempt}] POST {chat_url!r} | "
            f"requests timeout={_req_to}s (one value applies to connect+read) | "
            "not boto3 Bedrock in-notebook; HTTPS to your gateway only"
        )

        response = requests.post(chat_url, headers=headers, json=payload, timeout=_req_to)
        if response.status_code >= 400:
            body_text = response.text[:1000]
            print(f"[attempt {attempt}] HTTP {response.status_code} body:", body_text)
            if response.status_code == 504:
                _hk = ("Server", "Via", "x-amzn-RequestId", "x-amzn-ErrorType", "x-amz-apigw-id", "Date")
                _hd = {k: response.headers.get(k) for k in _hk if response.headers.get(k)}
                print(
                    f"[attempt {attempt}] 504: received HTTP response from server "
                    f"(not requests.exceptions.ReadTimeout at {_req_to}s). "
                    "Typical cause: API Gateway/Lambda/proxy integration timeout < model latency. "
                    f"Header subset: {_hd}"
                )
            last_error = RuntimeError(f"HTTP {response.status_code}: {body_text}")
            if response.status_code in (502, 503, 504, 529) and attempt < max_attempts:
                wait_s = min(PIPELINE_TRANSIENT_HTTP_SLEEP_CAP_S, PIPELINE_TRANSIENT_HTTP_SLEEP_MULT_S * attempt)
                print(f"[attempt {attempt}] transient HTTP {response.status_code}; sleeping {wait_s}s then retry")
                time.sleep(wait_s)
            continue

        result = response.json()

        print(f"[attempt {attempt}] max_tokens:", payload["data"]["max_tokens"])
        print(f"[attempt {attempt}] message_count:", len(payload["data"]["messages"]))
        print(f"[attempt {attempt}] result keys:", list(result.keys()) if isinstance(result, dict) else type(result).__name__)
        print(f"[attempt {attempt}] result.success:", result.get("success") if isinstance(result, dict) else None)
        print(f"[attempt {attempt}] repr(result.data):", repr(result.get("data")) if isinstance(result, dict) else None)

        if not isinstance(result, dict) or not result.get("success"):
            last_error = RuntimeError(f"Amplify returned failure: {result}")
            continue

        raw_data = result.get("data", "")
        if isinstance(raw_data, (dict, list)):
            raw_text = json.dumps(raw_data, ensure_ascii=False)
        else:
            raw_text = str(raw_data).strip()

        if not raw_text:
            last_error = RuntimeError(f"Empty response for {model_id}")
            continue

        if raw_text.lower().startswith("error:"):
            last_error = RuntimeError(raw_text)
            continue

        parsed_json = safe_parse_json(raw_text) if parse_json_lenient else parse_model_json_strict(raw_text)
        if parsed_json is None:
            last_error = RuntimeError(f"Strict mode: invalid JSON for {model_id}")
            continue

        if isinstance(parsed_json, dict) and "segments" in parsed_json:
            segments = parsed_json.get("segments", [])
        elif isinstance(parsed_json, list):
            segments = parsed_json
        else:
            segments = []

        if not isinstance(segments, list):
            segments = []

        segments = normalize_and_validate_segments(segments, csv_text, default_segment_label)
        if compact_justifications:
            segments = compact_segment_text(segments, default_label=default_segment_label)

        rule_based = False

        out_obj = {"segments": segments}
        if output_dir:
            os.makedirs(output_dir, exist_ok=True)
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(out_obj, f, indent=2, ensure_ascii=False)

        return {
            "json_path": json_path,
            "segment_count": len(segments),
            "rule_based": rule_based,
        }

    if last_error is not None:
        raise last_error
    raise RuntimeError(f"run_pipeline failed for {model_id} after {max_attempts} attempt(s)")


# Delegate for optional thin wrapper in the next cell — refreshed every time you re-run this cell.
_run_pipeline_original = run_pipeline


In [12]:
if "run_pipeline" not in globals():
    raise RuntimeError("Run Unified Inference Pipeline first (defines run_pipeline).")

if "_run_pipeline_original" not in globals():
    raise RuntimeError(
        "Re-run Unified Inference Pipeline to the end — it sets `_run_pipeline_original = run_pipeline`."
    )


def run_pipeline(model_id: str, messages: list, base_url: str, headers: dict,
                 output_dir: str = ".", output_stem: str | None = None,
                 max_tokens: int = 4000, temperature: float = 0,
                 max_attempts: int = 6,
                 strict_mode: bool = True,
                 segments_only: bool = True,
                 request_timeout: float = 600,
                 model_object: dict | None = None,
                 parse_json_lenient: bool = False,
                 default_segment_label: str = "ENACTING",
                 compact_justifications: bool = False):
    """Thin wrapper: delegates to globals()[\"_run_pipeline_original\"] each call (Pipeline cell refreshes it)."""

    def _call(mtok: int):
        impl = globals()["_run_pipeline_original"]
        return impl(
            model_id=model_id,
            messages=messages,
            base_url=base_url,
            headers=headers,
            output_dir=output_dir,
            output_stem=output_stem,
            max_tokens=mtok,
            temperature=temperature,
            max_attempts=max_attempts,
            strict_mode=strict_mode,
            segments_only=segments_only,
            request_timeout=request_timeout,
            model_object=model_object,
            parse_json_lenient=parse_json_lenient,
            default_segment_label=default_segment_label,
            compact_justifications=compact_justifications,
        )

    mtok0 = int(max_tokens)
    try:
        return _call(mtok0)
    except RuntimeError as e:
        msg = str(e)
        if strict_mode and (not parse_json_lenient) and ("Strict mode: invalid JSON" in msg):
            retry_tokens = min(max(mtok0 * 2, 16000), 128000)
            if retry_tokens > mtok0:
                return _call(retry_tokens)
        raise


print("Hotfix cell: thin run_pipeline wrapper — re-run Unified Inference Pipeline after editing the base run_pipeline.")


Hotfix cell: thin run_pipeline wrapper — re-run Unified Inference Pipeline after editing the base run_pipeline.


In [13]:
print("run_pipeline defined from:", run_pipeline.__code__.co_filename)
print("normalize_output_data exists:", "normalize_output_data" in globals())

run_pipeline defined from: /var/folders/kz/_0d2ww4n3wx819j1cz_zzvtw0000gn/T/ipykernel_4105/653080213.py
normalize_output_data exists: True


## Unified multi-behavior model pipeline

Run one configurable batch across any subset of the five behaviors (`enacting`, `interacting`, `monitoring`, `planning`, `reflecting`).

You only maintain input files in `PIPELINE_INPUTS`; this runner reuses each branch's prebuilt `messages_*` and metadata (`output_stem`, label).


In [14]:
# Unified runner for ENACTING/INTERACTING/MONITORING/PLANNING/REFLECTING.
# Prereqs: run branch cells first so BRANCH_* + messages_* are defined, and run pipeline cells.
RUN_MULTI_BEHAVIOR_BATCH = True  # set True to execute

BATCH_BEHAVIORS = ["enacting"]  # e.g. add "interacting", "monitoring", ... for full batch

# API cost: BATCH_MODELS_SINGLE = one id to run only that model; None = use full BATCH_MODELS list.
BATCH_MODELS_SINGLE = None
# Skip models already completed (case-insensitive id match). Use [] when nothing to skip.
BATCH_MODELS_SKIP = ["gpt-5.2"]

BATCH_MODELS = [
    "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    "o3",
    "gpt-5.2",
    "us.anthropic.claude-haiku-4-5-20251001-v1:0",
]
if BATCH_MODELS_SINGLE:
    _solo = str(BATCH_MODELS_SINGLE).strip()
    _solo_l = _solo.lower()
    _all_models = list(BATCH_MODELS)
    BATCH_MODELS = [m for m in BATCH_MODELS if isinstance(m, str) and m.strip().lower() == _solo_l]
    if not BATCH_MODELS:
        raise ValueError(
            f"BATCH_MODELS_SINGLE={_solo!r} matched none of configured models: {_all_models}"
        )
    print("BATCH_MODELS_SINGLE active -> models:", BATCH_MODELS)

if BATCH_MODELS_SKIP:
    _skip_l = {str(x).strip().lower() for x in BATCH_MODELS_SKIP if isinstance(x, str) and str(x).strip()}
    _before_skip = list(BATCH_MODELS)
    BATCH_MODELS = [m for m in BATCH_MODELS if isinstance(m, str) and m.strip().lower() not in _skip_l]
    if not BATCH_MODELS:
        raise ValueError(
            f"BATCH_MODELS_SKIP removed all models; before={_before_skip!r}, skip={BATCH_MODELS_SKIP!r}"
        )
    print("BATCH_MODELS_SKIP active -> models:", BATCH_MODELS)

BATCH_MAX_TOKENS = int(globals().get("MAX_TOKENS", 32000))
BATCH_MAX_ATTEMPTS = int(globals().get("MAX_ATTEMPTS", globals().get("PIPELINE_DEFAULT_MAX_ATTEMPTS", 6)))
BATCH_OUTPUT_DIR = ""

if not RUN_MULTI_BEHAVIOR_BATCH:
    print("Skip multi-behavior batch: set RUN_MULTI_BEHAVIOR_BATCH = True to run.")
else:
    if "get_available_models" not in globals() or "run_pipeline" not in globals():
        raise RuntimeError("Run BASE_URL/headers and Unified Inference Pipeline cells first.")

    behavior_registry = {
        "enacting": {
            "branch": "BRANCH_ENACTING",
            "messages": "messages",
        },
        "interacting": {
            "branch": "BRANCH_INTERACTING",
            "messages": "messages_interacting",
        },
        "monitoring": {
            "branch": "BRANCH_MONITORING",
            "messages": "messages_monitoring",
        },
        "planning": {
            "branch": "BRANCH_PLANNING",
            "messages": "messages_planning",
        },
        "reflecting": {
            "branch": "BRANCH_REFLECTING",
            "messages": "messages_reflecting",
        },
    }

    available_model_ids, default_model_id, _payload, available_model_lookup = get_available_models(BASE_URL, headers)
    offered_ids_lower = {m.lower().strip() for m in available_model_ids if isinstance(m, str)}

    reports_all = []
    for behavior in BATCH_BEHAVIORS:
        key = str(behavior).strip().lower()
        if key not in behavior_registry:
            print("Skip unknown behavior:", behavior)
            continue

        reg = behavior_registry[key]
        branch_obj = globals().get(reg["branch"])
        msg_list = globals().get(reg["messages"])
        if not isinstance(branch_obj, dict) or not isinstance(msg_list, list):
            raise RuntimeError(
                f"Missing {reg['branch']} or {reg['messages']}. Run the {key.upper()} branch cell first."
            )

        stem_prefix = branch_obj["output_stem"]
        default_label = branch_obj["behavior_label"]

        print(f"\n===== {key.upper()} =====")
        for mid in BATCH_MODELS:
            if not isinstance(mid, str) or mid.lower().strip() not in offered_ids_lower:
                print("Skip model (not offered):", mid)
                continue

            _catalog_model = available_model_lookup.get(mid)
            if not isinstance(_catalog_model, dict):
                _catalog_model = None
            if _catalog_model is None:
                print("[batch] no /available_models dict for", repr(mid), "-> run_pipeline uses minimal {id: ...}")

            if "claude-sonnet" in mid:
                model_tag = "Claude_sonnet_4_5"
            elif "claude-haiku" in mid:
                model_tag = "Claude_haiku_4_5"
            else:
                model_tag = mid.replace("-", "_").replace(".", "_")

            out_stem = f"{stem_prefix}_{model_tag}"
            print(f"\n=== {out_stem} ({mid}) ===")
            r = run_pipeline(
                model_id=mid,
                messages=msg_list,
                base_url=BASE_URL,
                headers=headers,
                output_dir=BATCH_OUTPUT_DIR,
                output_stem=out_stem,
                max_tokens=BATCH_MAX_TOKENS,
                temperature=0,
                max_attempts=BATCH_MAX_ATTEMPTS,
                strict_mode=True,
                segments_only=True,
                model_object=_catalog_model,
                parse_json_lenient=False,
                default_segment_label=default_label,
            )
            reports_all.append({"behavior": key, **r})
            print("Saved:", r["json_path"], "segments:", r.get("segment_count"))

    reports_all


BATCH_MODELS_SKIP active -> models: ['us.anthropic.claude-sonnet-4-5-20250929-v1:0', 'o3', 'us.anthropic.claude-haiku-4-5-20251001-v1:0']

===== ENACTING =====

=== ENACTING_L_I_Student_Mastersheet_EN_MESSI_D2_Claude_sonnet_4_5 (us.anthropic.claude-sonnet-4-5-20250929-v1:0) ===
[run_pipeline] options.model dict ['id', 'name', 'description', 'inputContextWindow', 'outputTokenLimit', 'supportsImages', 'supportsVideo', 'supportsImageGeneration', 'supportsReasoning', 'provider', 'supportsSystemPrompts', 'systemPrompt', 'inputTokenCost', 'outputTokenCost', 'inputCachedTokenCost', 'inputWriteCachedTokenCost']
[attempt 1] HTTP 504 body: {"message": "Endpoint request timed out"}
[attempt 1] transient HTTP 504; sleeping 20s then retry


KeyboardInterrupt: 

In [ ]:
# Summary table for all behavior x model runs.
# Run this AFTER the unified runner cell so `reports_all` is available.

import pandas as pd

if "reports_all" not in globals() or not isinstance(reports_all, list) or len(reports_all) == 0:
    raise RuntimeError("`reports_all` is empty. Run the unified runner cell first.")

rows = []
for r in reports_all:
    if not isinstance(r, dict):
        continue
    behavior = str(r.get("behavior", "")).strip().lower()
    json_path = str(r.get("json_path", "") or "")
    seg = r.get("segment_count", None)

    model_id = r.get("model_id")
    if not model_id:
        p = json_path.rsplit("/", 1)[-1]
        if "Claude_sonnet_4_5" in p:
            model_id = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
        elif "Claude_haiku_4_5" in p:
            model_id = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
        elif "gpt_5_2" in p or "ChatGPT_5_2" in p:
            model_id = "gpt-5.2"
        elif p.endswith("_o3.json"):
            model_id = "o3"
        else:
            model_id = ""

    rows.append({
        "behavior": behavior,
        "model_id": str(model_id),
        "output_file": json_path.rsplit("/", 1)[-1],
        "segment_count": seg,
        "json_path": json_path,
        "status": "ok" if seg is not None else "unknown",
    })

raw_df = pd.DataFrame(rows)

behaviors = [str(x).strip().lower() for x in globals().get("BATCH_BEHAVIORS", ["enacting", "interacting", "monitoring", "planning", "reflecting"])]
models = [str(x).strip() for x in globals().get("BATCH_MODELS", [
    "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    "o3",
    "gpt-5.2",
    "us.anthropic.claude-haiku-4-5-20251001-v1:0",
])]

full_index = pd.MultiIndex.from_product([behaviors, models], names=["behavior", "model_id"])
full_df = full_index.to_frame(index=False).merge(raw_df, on=["behavior", "model_id"], how="left")
full_df["status"] = full_df["status"].fillna("missing")

print("===== FULL TABLE: each behavior x each model =====")
display(full_df[["behavior", "model_id", "output_file", "segment_count", "status", "json_path"]])

print("===== PIVOT TABLE: segment_count =====")
pivot_df = full_df.pivot(index="behavior", columns="model_id", values="segment_count")
pivot_df = pivot_df.reindex(behaviors)
display(pivot_df)


## Code-only Boundary Baseline (no LLM)

Use deterministic boundary rules on source branch CSVs to get a rule-based segment count table for comparison with model outputs.

In [ ]:
# Rule-only baseline: no model call, boundary-condition heuristics only.
# Produces a table shaped like the model pivot for side-by-side comparison.

import re
import pandas as pd


def _parse_ts(ts):
    s = str(ts or "").strip()
    if not s:
        return None
    parts = s.split(":")
    try:
        if len(parts) == 3:
            h, m, sec = parts
            return int(h) * 3600 + int(m) * 60 + float(sec)
        if len(parts) == 2:
            m, sec = parts
            return int(m) * 60 + float(sec)
    except Exception:
        return None
    return None


def _norm_df(d):
    d = d.copy().fillna("")
    for c in ["start_time", "end_time", "modality", "data"]:
        if c not in d.columns:
            d[c] = ""
    d["start_s"] = d["start_time"].map(_parse_ts)
    d["end_s"] = d["end_time"].map(_parse_ts)
    d["modality"] = d["modality"].astype(str)
    d["data"] = d["data"].astype(str)
    # Instant events: use start as end.
    d.loc[d["end_s"].isna(), "end_s"] = d.loc[d["end_s"].isna(), "start_s"]
    d = d.dropna(subset=["start_s", "end_s"])
    d.loc[d["end_s"] < d["start_s"], "end_s"] = d.loc[d["end_s"] < d["start_s"], "start_s"]
    return d


def _interval_overlap(a0, a1, b0, b1):
    s = max(a0, b0)
    e = min(a1, b1)
    return (s, e) if e >= s else None


def _merge_intervals(iv):
    # Boundary-only mode: normalize + sort, NO merging of adjacent/overlapping intervals.
    norm = []
    for x in iv:
        if not isinstance(x, (list, tuple)) or len(x) != 2:
            continue
        s, e = x[0], x[1]
        if s is None or e is None:
            continue
        s = float(s)
        e = float(e)
        if e < s:
            s, e = e, s
        norm.append((s, e))

    norm = sorted(norm, key=lambda z: (z[0], z[1]))

    # De-duplicate exact repeats only; keep fragments as-is.
    out = []
    seen = set()
    for s, e in norm:
        k = (s, e)
        if k in seen:
            continue
        seen.add(k)
        out.append((s, e))
    return out


def code_segments_enacting(df):
    d = _norm_df(df)
    gaze = d[(d["modality"] == "gaze") & (d["data"] == "Screen")][["start_s", "end_s"]].values.tolist()
    mov = d[(d["modality"] == "movement") & (d["data"] == "moving")][["start_s", "end_s"]].values.tolist()
    act = d[d["modality"] == "action"]["start_s"].tolist()
    sta = d[d["modality"] == "state"]["start_s"].tolist()

    segs = []
    for gs, ge in gaze:
        for ms, me in mov:
            ov = _interval_overlap(gs, ge, ms, me)
            if ov:
                segs.append(ov)
        for t in act + sta:
            if gs <= t <= ge:
                segs.append((gs, t))
    return _merge_intervals(segs)


def code_segments_monitoring(df, state_window=3.0):
    d = _norm_df(df)
    gaze = d[(d["modality"] == "gaze") & (d["data"] == "Screen")][["start_s", "end_s"]].values.tolist()
    stat = d[(d["modality"] == "movement") & (d["data"] == "stationary")][["start_s", "end_s"]].values.tolist()
    state_ts = d[d["modality"] == "state"]["start_s"].tolist()

    segs = []
    for gs, ge in gaze:
        for ss, se in stat:
            ov = _interval_overlap(gs, ge, ss, se)
            if not ov:
                continue
            os, oe = ov
            hit = any((os - state_window) <= t <= (oe + state_window) for t in state_ts)
            if hit:
                segs.append((os, oe))
    return _merge_intervals(segs)


def _speech_rows(df):
    d = _norm_df(df)
    return d[d["modality"] == "speech"].copy(), d


def code_segments_interacting(df):
    sp, d = _speech_rows(df)
    # C1 surrogate: oxygen/sugar state transitions.
    c1 = d[(d["modality"] == "state") & (d["data"].isin(["oxygen", "sugar"]))][["start_s", "end_s"]].values.tolist()

    # C2 surrogate: speech while looking at another student.
    student_gaze = d[(d["modality"] == "gaze") & (d["data"].str.contains(r"Student\s*-", regex=True))][["start_s", "end_s"]].values.tolist()
    c2 = []
    for _, r in sp.iterrows():
        for gs, ge in student_gaze:
            ov = _interval_overlap(r["start_s"], r["end_s"], gs, ge)
            if ov:
                c2.append(ov)
                break
    return _merge_intervals(c1 + c2)


def _group_with_gap(intervals, gap=10.0):
    # Boundary-only mode: keep each speech boundary interval, no gap-based grouping.
    norm = []
    for x in intervals:
        if not isinstance(x, (list, tuple)) or len(x) != 2:
            continue
        s, e = x[0], x[1]
        if s is None or e is None:
            continue
        s = float(s)
        e = float(e)
        if e < s:
            s, e = e, s
        norm.append((s, e))
    norm = sorted(norm, key=lambda z: (z[0], z[1]))

    out = []
    seen = set()
    for s, e in norm:
        k = (s, e)
        if k in seen:
            continue
        seen.add(k)
        out.append((s, e))
    return out


def code_segments_reflecting(df):
    sp, _ = _speech_rows(df)
    refl_pat = re.compile(r"\b(why|because|thought|happen|happened|meant|i think|was|were)\b", re.I)
    iv = []
    for _, r in sp.iterrows():
        if refl_pat.search(str(r["data"])):
            iv.append((r["start_s"], r["end_s"]))
    return _group_with_gap(iv, gap=10.0)


def code_segments_planning(df):
    sp, _ = _speech_rows(df)
    plan_pat = re.compile(r"\b(gonna|going to|need to|have to|should|next|i will|i'll|we can|let's)\b", re.I)
    iv = []
    for _, r in sp.iterrows():
        if plan_pat.search(str(r["data"])):
            iv.append((r["start_s"], r["end_s"]))
    return _group_with_gap(iv, gap=10.0)


# Build code-only baseline counts from branch dataframes.
branch_by_behavior = {
    "enacting": globals().get("BRANCH_ENACTING"),
    "interacting": globals().get("BRANCH_INTERACTING"),
    "monitoring": globals().get("BRANCH_MONITORING"),
    "planning": globals().get("BRANCH_PLANNING"),
    "reflecting": globals().get("BRANCH_REFLECTING"),
}

for b, obj in branch_by_behavior.items():
    if not isinstance(obj, dict) or "df_model" not in obj:
        raise RuntimeError(f"Missing {b} branch dataframe. Run branch cells first.")

code_counts = {
    "enacting": len(code_segments_enacting(branch_by_behavior["enacting"]["df_model"])),
    "interacting": len(code_segments_interacting(branch_by_behavior["interacting"]["df_model"])),
    "monitoring": len(code_segments_monitoring(branch_by_behavior["monitoring"]["df_model"])),
    "planning": len(code_segments_planning(branch_by_behavior["planning"]["df_model"])),
    "reflecting": len(code_segments_reflecting(branch_by_behavior["reflecting"]["df_model"])),
}

code_df = pd.DataFrame([
    {"behavior": k, "model_id": "code_boundary_rules", "segment_count": v}
    for k, v in code_counts.items()
])

print("===== CODE-ONLY BASELINE TABLE =====")
display(code_df.sort_values("behavior"))

print("===== CODE-ONLY PIVOT (same shape) =====")
code_pivot_df = code_df.pivot(index="behavior", columns="model_id", values="segment_count")
code_pivot_df = code_pivot_df.reindex(["enacting", "interacting", "monitoring", "planning", "reflecting"])
display(code_pivot_df)

# Optional merged comparison with model pivot if available.
if "pivot_df" in globals() and isinstance(pivot_df, pd.DataFrame):
    print("===== MODEL + CODE COMPARISON PIVOT =====")
    merged = pivot_df.copy()
    merged["code_boundary_rules"] = code_pivot_df["code_boundary_rules"]
    display(merged)


## time inverter


In [ ]:
import os
COMPARE_BASE_DIR = ""
print("COMPARE_BASE_DIR:", COMPARE_BASE_DIR)

In [ ]:
import json
import os
import time
from datetime import timedelta


def time_to_seconds(time_str):
    if ":" in time_str:
        h, m, s = time_str.split(":")
        return int(h) * 3600 + int(m) * 60 + float(s)
    else:
        return float(time_str)


def seconds_to_tc(seconds):
    td = timedelta(seconds=seconds)
    total_seconds = int(td.total_seconds())
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    secs = total_seconds % 60
    return f"{hours:02}:{minutes:02}:{secs:02}.0000"


def clean_segments(segments):
    processed = []

    for seg in segments:
        start = time_to_seconds(seg["time_in"])
        end = time_to_seconds(seg["time_out"])

        processed.append((start, end))

    processed.sort(key=lambda x: x[0])

    return processed


def convert_llm_to_amalia(input_file, output_file, metadata_id, label):

    with open(input_file, "r") as f:
        data = json.load(f)

    raw_segments = data["segments"]
    cleaned = clean_segments(raw_segments)

    # Build level1 list
    level1_segments = []

    for start, end in cleaned:
        level1_segments.append({
            "tclevel": 1,
            "tcin": seconds_to_tc(start),
            "tcout": seconds_to_tc(end),
            "label": label
        })

    amalia_format = {
        "type": "text",
        "id": metadata_id,
        "algorithm": "LLM",
        "processor": "ChatGPT",
        "processed": int(time.time() * 1000),
        "version": 1,
        "localisation": [
            {
                "sublocalisations": {
                    "localisation": level1_segments
                },
                "type": "text",
                "tcin": "00:00:00.0000",
                "tcout": "00:10:27.0000",
                "tclevel": 0
            }
        ]
    }

    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    with open(output_file, "w") as f:
        json.dump(amalia_format, f, indent=4)

    print("Generated:", output_file)
    print("Segments after merge:", len(cleaned))


if __name__ == "__main__":

    BASE_DIR = os.getcwd()

    _branch = globals().get("BRANCH_ENACTING")
    if not isinstance(_branch, dict):
        raise RuntimeError("Run ENACTING branch cells first (defines BRANCH_ENACTING).")

    _P = str(_branch["output_stem"])
    _proc = str(_branch["processor_label"])

    jobs = [
        (
            f"{_P}_ChatGPT_5_2.json",
            "samples-data/data-final/d2g2-segments-messi-gpt52-enact.json",
            "d2g2-segments-messi-gpt52-enact",
            _proc,
        ),
        (
            f"{_P}_o3.json",
            "samples-data/data-final/d2g2-segments-messi-o3-enact.json",
            "d2g2-segments-messi-o3-enact",
            _proc,
        ),
        (
            f"{_P}_Claude_haiku_4_5.json",
            "samples-data/data-final/d2g2-segments-messi-claude-haiku-4-5-enact.json",
            "d2g2-segments-messi-claude-haiku-4-5-enact",
            _proc,
        ),
        (
            f"{_P}_Claude_sonnet_4_5.json",
            "samples-data/data-final/d2g2-segments-messi-claude-sonnet-4-5-enact.json",
            "d2g2-segments-messi-claude-sonnet-4-5-enact",
            _proc,
        ),
    ]

    for input_file, output_file, metadata_id, label in jobs:
        input_path = os.path.join(BASE_DIR, input_file)
        output_path = os.path.join(BASE_DIR, output_file)

        if not os.path.exists(input_path):
            print(f"Skip missing file: {input_path}")
            continue

        convert_llm_to_amalia(input_path, output_path, metadata_id, label)


## Segment Boundary Comparison (Merged)
Merged from `segment_boundary_comparison.ipynb`.

In [ ]:
# Unified runner (strict): fail fast, same token budget for every model.
RUN_MULTI_BEHAVIOR_BATCH = True  # set True to execute

BATCH_BEHAVIORS = ["enacting"]  # e.g. add "interacting", "monitoring", ... for full batch

# API cost: BATCH_MODELS_SINGLE = one id to run only that model; None = use full BATCH_MODELS list.
BATCH_MODELS_SINGLE = None
# Skip models already completed (case-insensitive id match). Use [] when nothing to skip.
BATCH_MODELS_SKIP = ["gpt-5.2"]

BATCH_MODELS = [
    "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    "o3",
    "gpt-5.2",
    "us.anthropic.claude-haiku-4-5-20251001-v1:0",
]
if BATCH_MODELS_SINGLE:
    _solo = str(BATCH_MODELS_SINGLE).strip()
    _solo_l = _solo.lower()
    _all_models = list(BATCH_MODELS)
    BATCH_MODELS = [m for m in BATCH_MODELS if isinstance(m, str) and m.strip().lower() == _solo_l]
    if not BATCH_MODELS:
        raise ValueError(
            f"BATCH_MODELS_SINGLE={_solo!r} matched none of configured models: {_all_models}"
        )
    print("BATCH_MODELS_SINGLE active -> models:", BATCH_MODELS)

if BATCH_MODELS_SKIP:
    _skip_l = {str(x).strip().lower() for x in BATCH_MODELS_SKIP if isinstance(x, str) and str(x).strip()}
    _before_skip = list(BATCH_MODELS)
    BATCH_MODELS = [m for m in BATCH_MODELS if isinstance(m, str) and m.strip().lower() not in _skip_l]
    if not BATCH_MODELS:
        raise ValueError(
            f"BATCH_MODELS_SKIP removed all models; before={_before_skip!r}, skip={BATCH_MODELS_SKIP!r}"
        )
    print("BATCH_MODELS_SKIP active -> models:", BATCH_MODELS)

BATCH_MAX_TOKENS = int(globals().get("MAX_TOKENS", 32000))
BATCH_MAX_ATTEMPTS = int(globals().get("MAX_ATTEMPTS", globals().get("PIPELINE_DEFAULT_MAX_ATTEMPTS", 6)))
BATCH_OUTPUT_DIR = ""

if not RUN_MULTI_BEHAVIOR_BATCH:
    print("Skip multi-behavior batch: set RUN_MULTI_BEHAVIOR_BATCH = True to run.")
else:
    if "get_available_models" not in globals() or "run_pipeline" not in globals():
        raise RuntimeError("Run BASE_URL/headers and Unified Inference Pipeline cells first.")

    behavior_registry = {
        "enacting": {"branch": "BRANCH_ENACTING", "messages": "messages"},
        "interacting": {"branch": "BRANCH_INTERACTING", "messages": "messages_interacting"},
        "monitoring": {"branch": "BRANCH_MONITORING", "messages": "messages_monitoring"},
        "planning": {"branch": "BRANCH_PLANNING", "messages": "messages_planning"},
        "reflecting": {"branch": "BRANCH_REFLECTING", "messages": "messages_reflecting"},
    }

    available_model_ids, default_model_id, _payload, available_model_lookup = get_available_models(BASE_URL, headers)
    offered_ids_lower = {m.lower().strip() for m in available_model_ids if isinstance(m, str)}

    reports_all = []
    for behavior in BATCH_BEHAVIORS:
        key = str(behavior).strip().lower()
        if key not in behavior_registry:
            raise RuntimeError(f"Unknown behavior in BATCH_BEHAVIORS: {behavior}")

        reg = behavior_registry[key]
        branch_obj = globals().get(reg["branch"])
        msg_list = globals().get(reg["messages"])
        if not isinstance(branch_obj, dict) or not isinstance(msg_list, list):
            raise RuntimeError(f"Missing {reg['branch']} or {reg['messages']}. Run {key.upper()} branch cell first.")

        stem_prefix = branch_obj["output_stem"]
        default_label = branch_obj["behavior_label"]

        print(f"\n===== {key.upper()} =====")
        for mid in BATCH_MODELS:
            if not isinstance(mid, str) or mid.lower().strip() not in offered_ids_lower:
                raise RuntimeError(f"Model not offered by API: {mid}")

            _catalog_model = available_model_lookup.get(mid)
            if not isinstance(_catalog_model, dict):
                _catalog_model = None
            if _catalog_model is None:
                print("[batch] no /available_models dict for", repr(mid), "-> run_pipeline uses minimal {id: ...}")

            if "claude-sonnet" in mid:
                model_tag = "Claude_sonnet_4_5"
            elif "claude-haiku" in mid:
                model_tag = "Claude_haiku_4_5"
            else:
                model_tag = mid.replace("-", "_").replace(".", "_")

            out_stem = f"{stem_prefix}_{model_tag}"
            print(f"\n=== {out_stem} ({mid}) ===")
            print("max_tokens:", BATCH_MAX_TOKENS)

            r = run_pipeline(
                model_id=mid,
                messages=msg_list,
                base_url=BASE_URL,
                headers=headers,
                output_dir=BATCH_OUTPUT_DIR,
                output_stem=out_stem,
                max_tokens=BATCH_MAX_TOKENS,
                temperature=0,
                max_attempts=BATCH_MAX_ATTEMPTS,
                strict_mode=True,
                segments_only=True,
                model_object=_catalog_model,
                parse_json_lenient=False,
                default_segment_label=default_label,
            )
            reports_all.append({"behavior": key, **r})
            print("Saved:", r["json_path"], "segments:", r.get("segment_count"))

    print("\nDone. success:", len(reports_all), "failed: 0")
    reports_all


In [ ]:
# moved above `time inverter`

import pandas as pd

if "reports_all" not in globals() or not isinstance(reports_all, list) or len(reports_all) == 0:
    raise RuntimeError("`reports_all` is empty. Run the unified runner cell first.")

rows = []
for r in reports_all:
    if not isinstance(r, dict):
        continue
    behavior = str(r.get("behavior", "")).strip().lower()
    json_path = str(r.get("json_path", "") or "")
    seg = r.get("segment_count", None)

    model_id = r.get("model_id")
    if not model_id:
        p = json_path.rsplit("/", 1)[-1]
        if "Claude_sonnet_4_5" in p:
            model_id = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
        elif "Claude_haiku_4_5" in p:
            model_id = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
        elif "gpt_5_2" in p or "ChatGPT_5_2" in p:
            model_id = "gpt-5.2"
        elif p.endswith("_o3.json"):
            model_id = "o3"
        else:
            model_id = ""

    rows.append({
        "behavior": behavior,
        "model_id": str(model_id),
        "output_file": json_path.rsplit("/", 1)[-1],
        "segment_count": seg,
        "json_path": json_path,
        "status": "ok" if seg is not None else "unknown",
    })

raw_df = pd.DataFrame(rows)

behaviors = [str(x).strip().lower() for x in globals().get("BATCH_BEHAVIORS", ["enacting", "interacting", "monitoring", "planning", "reflecting"])]
models = [str(x).strip() for x in globals().get("BATCH_MODELS", [
    "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    "o3",
    "gpt-5.2",
    "us.anthropic.claude-haiku-4-5-20251001-v1:0",
])]

full_index = pd.MultiIndex.from_product([behaviors, models], names=["behavior", "model_id"])
full_df = full_index.to_frame(index=False).merge(raw_df, on=["behavior", "model_id"], how="left")
full_df["status"] = full_df["status"].fillna("missing")

print("===== FULL TABLE: each behavior x each model =====")
display(full_df[["behavior", "model_id", "output_file", "segment_count", "status", "json_path"]])

print("===== PIVOT TABLE: segment_count =====")
pivot_df = full_df.pivot(index="behavior", columns="model_id", values="segment_count")
pivot_df = pivot_df.reindex(behaviors)
display(pivot_df)


In [ ]:
import os
COMPARE_BASE_DIR = ""
os.path.abspath(COMPARE_BASE_DIR)

In [ ]:
with open(os.path.join(COMPARE_BASE_DIR, "samples-data/data-final/d2g2-segments-messi-gpt52-enact.json")) as f:
    data = json.load(f)

print(type(data))
print(data)


In [ ]:
# Method1: Multi-model all-pairs comparison
import json
import itertools
from collections import Counter
import pandas as pd


def classify(a_start, a_end, b_start, b_end):
    if a_start == b_start and a_end == b_end:
        return "EXACT_MATCH"
    if a_start == b_start:
        return "SAME_START"
    if a_end == b_end:
        return "SAME_END"
    if b_start <= a_start and b_end >= a_end:
        return "B_CONTAINS_A"
    if a_start <= b_start and a_end >= b_end:
        return "A_CONTAINS_B"
    if a_start < b_end and a_end > b_start:
        return "PARTIAL_OVERLAP"
    return "NO_OVERLAP"


def tc_to_seconds(tc):
    h, m, s = tc.split(":")
    return int(h) * 3600 + int(m) * 60 + float(s)


MODEL_FILES = {
    "gpt52": "samples-data/data-final/d2g2-segments-messi-gpt52-enact.json",
    "o3": "samples-data/data-final/d2g2-segments-messi-o3-enact.json",
    "claude_haiku": "samples-data/data-final/d2g2-segments-messi-claude-haiku-4-5-enact.json",
    "claude_sonnet": "samples-data/data-final/d2g2-segments-messi-claude-sonnet-4-5-enact.json",
}

model_segments = {}
for model_name, rel_path in MODEL_FILES.items():
    abs_path = os.path.join(COMPARE_BASE_DIR, rel_path)
    if not os.path.exists(abs_path):
        print(f"Skip missing file for {model_name}: {abs_path}")
        continue
    with open(abs_path) as f:
        data = json.load(f)
    segs = data["localisation"][0]["sublocalisations"]["localisation"]
    model_segments[model_name] = segs

print("Loaded models and segment counts:")
for k, v in model_segments.items():
    print(f"- {k}: {len(v)}")

pairwise_relation_stats = []
pairwise_results = {}

for model_a, model_b in itertools.combinations(model_segments.keys(), 2):
    results = []
    segs_a = model_segments[model_a]
    segs_b = model_segments[model_b]

    for seg_a in segs_a:
        a_start = tc_to_seconds(seg_a["tcin"])
        a_end = tc_to_seconds(seg_a["tcout"])
        for seg_b in segs_b:
            b_start = tc_to_seconds(seg_b["tcin"])
            b_end = tc_to_seconds(seg_b["tcout"])
            rel = classify(a_start, a_end, b_start, b_end)
            results.append({
                "model_a": model_a,
                "model_b": model_b,
                "a": [a_start, a_end],
                "b": [b_start, b_end],
                "relation": rel,
            })

    pair_key = f"{model_a}__vs__{model_b}"
    pairwise_results[pair_key] = results

    c = Counter(r["relation"] for r in results)
    row = {"pair": pair_key, "total": len(results)}
    row.update(c)
    pairwise_relation_stats.append(row)

pairwise_stats_df = pd.DataFrame(pairwise_relation_stats).fillna(0)
pairwise_stats_df = pairwise_stats_df.sort_values("pair").reset_index(drop=True)
pairwise_stats_df


In [ ]:
# Optional: inspect one pair in detail
PAIR_TO_INSPECT = "gpt52__vs__claude_sonnet"

if PAIR_TO_INSPECT in pairwise_results:
    sample = pairwise_results[PAIR_TO_INSPECT][:20]
    print(f"Pair: {PAIR_TO_INSPECT} | sample rows: {len(sample)}")
    for r in sample:
        print(r)
else:
    print(f"Pair not found: {PAIR_TO_INSPECT}")


In [ ]:
# Method2: Best-match mapping for any chosen pair
MODEL_A = "gpt52"
MODEL_B = "claude_sonnet"


def overlap(a_start, a_end, b_start, b_end):
    return max(0, min(a_end, b_end) - max(a_start, b_start))


best_matches = []

if MODEL_A in model_segments and MODEL_B in model_segments:
    segs_a = model_segments[MODEL_A]
    segs_b = model_segments[MODEL_B]

    for i, seg_a in enumerate(segs_a):
        a_start = tc_to_seconds(seg_a["tcin"])
        a_end = tc_to_seconds(seg_a["tcout"])

        best_overlap = -1
        best_seg = None
        best_j = None

        for j, seg_b in enumerate(segs_b):
            b_start = tc_to_seconds(seg_b["tcin"])
            b_end = tc_to_seconds(seg_b["tcout"])
            ov = overlap(a_start, a_end, b_start, b_end)
            if ov > best_overlap:
                best_overlap = ov
                best_seg = (b_start, b_end)
                best_j = j + 1

        relation = classify(a_start, a_end, best_seg[0], best_seg[1]) if best_seg else "NO_MATCH"
        best_matches.append({
            "A_model": MODEL_A,
            "A_id": i + 1,
            "A_segment": [a_start, a_end],
            "B_model": MODEL_B,
            "B_id": best_j,
            "B_segment": best_seg,
            "best_overlap": best_overlap,
            "relation": relation,
        })

len(best_matches)


In [ ]:
print("Total best matches:", len(best_matches))
for m in best_matches[:30]:
    print(m)

In [ ]:
# Method3: One-to-many overlap mapping for any chosen pair
one_to_many = []

if MODEL_A in model_segments and MODEL_B in model_segments:
    segs_a = model_segments[MODEL_A]
    segs_b = model_segments[MODEL_B]

    for i, seg_a in enumerate(segs_a):
        a_start = tc_to_seconds(seg_a["tcin"])
        a_end = tc_to_seconds(seg_a["tcout"])

        related_segments = []
        for j, seg_b in enumerate(segs_b):
            b_start = tc_to_seconds(seg_b["tcin"])
            b_end = tc_to_seconds(seg_b["tcout"])
            relation = classify(a_start, a_end, b_start, b_end)
            if relation != "NO_OVERLAP":
                related_segments.append({
                    "B_id": j + 1,
                    "segment": [b_start, b_end],
                    "relation": relation,
                })

        one_to_many.append({
            "A_id": i + 1,
            "A_segment": [a_start, a_end],
            "related_B": related_segments,
        })

len(one_to_many)

In [ ]:
for r in one_to_many[:3]:
    print(f"{MODEL_A}_", r["A_id"], r["A_segment"])
    for seg in r["related_B"]:
        print(f"   → {MODEL_B}_", seg["B_id"], seg["segment"], seg["relation"])

In [ ]:
# Prep for Method4 visualization (multi-model)
import matplotlib.pyplot as plt

boundary_data = {}
for model_name, segs in model_segments.items():
    starts = [tc_to_seconds(seg["tcin"]) for seg in segs]
    ends = [tc_to_seconds(seg["tcout"]) for seg in segs]
    boundary_data[model_name] = {"starts": starts, "ends": ends}

boundary_data.keys()

In [ ]:
## Method4: Segment boundary alignment visualization (multi-model)
plt.figure(figsize=(14, 6))

model_order = list(boundary_data.keys())
if not model_order:
    raise ValueError("No model data loaded for visualization.")

row_gap = 1.2
for i, model_name in enumerate(model_order):
    y_start = (len(model_order) - i) * row_gap
    y_end = y_start - 0.45

    starts = boundary_data[model_name]["starts"]
    ends = boundary_data[model_name]["ends"]

    plt.scatter(starts, [y_start] * len(starts), s=30, label=f"{model_name} start")
    plt.scatter(ends, [y_end] * len(ends), s=30, marker="x", label=f"{model_name} end")

plt.xlabel("Time (seconds)")
plt.title("Segment boundary comparison across models")

from matplotlib.ticker import MultipleLocator
ax = plt.gca()
ax.xaxis.set_major_locator(MultipleLocator(50))
ax.xaxis.set_minor_locator(MultipleLocator(10))
ax.grid(which="major", linestyle="-", linewidth=0.8, alpha=0.6)
ax.grid(which="minor", linestyle=":", linewidth=0.5, alpha=0.6)

plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0)
plt.tight_layout()
plt.show()

## 12. Code-only boundary detection (baseline segment counts)

Deterministic segments from the **written boundary rules** in each prompt (heuristic where the rubric needs interpretation, e.g. REFLECTING/PLANNING semantics). **No LLM.** Run after the **Unified Inference Pipeline** cell so `_parse_time_to_seconds` exists (or the cell defines a local fallback).

Uses the same branch CSV shaping style as **§5–§7** (monitoring/planning/reflecting) for baseline counts.

In [ ]:
import re
import pandas as pd

# --- time helpers (reuse pipeline if available) ---
if "_parse_time_to_seconds" not in globals():

    def _parse_time_to_seconds(time_str):
        if not isinstance(time_str, str):
            return None
        s = time_str.strip()
        if not s:
            return None
        parts = s.split(":")
        try:
            if len(parts) == 3:
                h, m, sec = parts
                return int(h) * 3600 + int(m) * 60 + float(sec)
            if len(parts) == 2:
                m, sec = parts
                return int(m) * 60 + float(sec)
        except ValueError:
            return None
        return None


def _iv_parse(row):
    s = _parse_time_to_seconds(str(row.get("start_time", "")))
    e_raw = str(row.get("end_time", "")).strip()
    e = _parse_time_to_seconds(e_raw) if e_raw else s
    if s is None:
        return None
    if e is None:
        e = s
    return (s, max(e, s))


def _merge_intervals(ivs, max_gap=0.0):
    ivs = sorted([x for x in ivs if x], key=lambda x: x[0])
    if not ivs:
        return []
    out = [list(ivs[0])]
    for s, e in ivs[1:]:
        if s <= out[-1][1] + max_gap:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return [(a, b) for a, b in out]


def _intersect(a, b):
    s = max(a[0], b[0])
    e = min(a[1], b[1])
    if s > e:
        return None
    return (s, e)


def _dedupe_segs(segs):
    seen = set()
    out = []
    for s, e, lab in segs:
        k = (round(s, 3), round(e, 3), lab)
        if k in seen:
            continue
        seen.add(k)
        out.append((s, e, lab))
    return out


def code_segments_enacting(df):
    d = df.fillna("")
    moving = [_iv_parse(r) for _, r in d[d["modality"] == "movement"].iterrows() if r.get("data") == "moving"]
    stat = [_iv_parse(r) for _, r in d[d["modality"] == "movement"].iterrows() if r.get("data") == "stationary"]
    screen = [_iv_parse(r) for _, r in d[d["modality"] == "gaze"].iterrows() if r.get("data") == "Screen"]
    segs = []
    for mv in moving:
        for gv in screen:
            hit = _intersect(mv, gv)
            if hit:
                segs.append((*hit, "C1_move+Screen"))
    for _, r in d[d["modality"] == "action"].iterrows():
        t = _parse_time_to_seconds(str(r.get("start_time", "")))
        if t is None:
            continue
        for gv in screen:
            if gv[0] <= t <= gv[1]:
                segs.append((gv[0], t, "C2_action+Screen"))
                break
    for _, r in d[d["modality"] == "state"].iterrows():
        t = _parse_time_to_seconds(str(r.get("start_time", "")))
        if t is None:
            continue
        for gv in screen:
            if gv[0] <= t <= gv[1]:
                segs.append((gv[0], t, "C3_state+Screen"))
                break
    return _dedupe_segs(segs)


def code_segments_monitoring(df, state_window=3.0):
    """Stationary + Screen gaze overlap, with a state change within ±state_window s of that overlap (prompt paraphrase)."""
    d = df.fillna("")
    screen = [_iv_parse(r) for _, r in d[d["modality"] == "gaze"].iterrows() if r.get("data") == "Screen"]
    states = []
    for _, r in d[d["modality"] == "state"].iterrows():
        t = _parse_time_to_seconds(str(r.get("start_time", "")))
        if t is not None:
            states.append(t)
    st_ivs = []
    for _, r in d[d["modality"] == "movement"].iterrows():
        if str(r.get("data", "")).strip().lower() != "stationary":
            continue
        iv = _iv_parse(r)
        if iv:
            st_ivs.append(iv)
    # Collapse brief moving gaps (≤1s) into one stationary spell, then merge adjacent stationary.
    st_ivs = _merge_intervals(st_ivs, max_gap=1.0)
    segs = []
    for st in st_ivs:
        st0, st1 = st
        for gv in screen:
            o = _intersect(st, gv)
            if not o:
                continue
            os, oe = o
            if any(
                abs(ts - x) <= state_window
                for ts in states
                for x in (st0, st1, os, oe)
            ):
                segs.append((os, oe, "MONITORING_rule"))
    return _dedupe_segs(segs)


def _speech_turns(df, speaker_name):
    d = df.fillna("")
    turns = []
    pat = re.compile(r"^\s*(Student|Teacher|Researcher)\s*-\s*([^:]+):\s*(.*)$", re.I)
    for _, r in d[d["modality"] == "speech"].iterrows():
        t = _parse_time_to_seconds(str(r.get("start_time", "")))
        if t is None:
            continue
        m = pat.match(str(r.get("data", "")))
        if not m:
            continue
        role, who, utt = m.group(1), m.group(2).strip(), m.group(3).strip()
        turns.append((t, role, who, utt))
    turns.sort(key=lambda x: x[0])
    return turns


def code_segments_interacting(df, speaker_name):
    d = df.fillna("")
    segs = []
    for _, r in d[d["modality"] == "state"].iterrows():
        st = str(r.get("data", "")).strip().lower()
        if st not in {"oxygen", "sugar"}:
            continue
        t = _parse_time_to_seconds(str(r.get("start_time", "")))
        if t is not None:
            segs.append((t, t, "INT_state_pair"))
    turns = _speech_turns(df, speaker_name)
    sn = (speaker_name or "").strip()
    for i, (t0, r0, w0, u0) in enumerate(turns):
        if r0.lower() != "student" or w0.strip() != sn:
            continue
        win = [x for x in turns if abs(x[0] - t0) <= 15.0]
        if any(x[1].lower() == "student" and x[2].strip() != sn for x in win):
            segs.append((t0, t0, "INT_peer_speech"))
    return _dedupe_segs(segs)


def code_segments_reflecting(df, speaker_name, gap=10.0):
    turns = _speech_turns(df, speaker_name)
    sn = (speaker_name or "").strip()
    stu = [(t, u) for t, r, w, u in turns if r.lower() == "student" and w.strip() == sn]
    if not stu:
        return []
    segs = []
    cur_s, cur_e, _ = stu[0][0], stu[0][0], stu[0][1]
    for t, _u in stu[1:]:
        if t - cur_e <= gap:
            cur_e = t
        else:
            segs.append((cur_s, cur_e, "REFL_speech_block"))
            cur_s = cur_e = t
    segs.append((cur_s, cur_e, "REFL_speech_block"))
    return _dedupe_segs(segs)


_PLAN_RE = re.compile(
    r"\b('ll|will|gonna|going to|let's|lets|need to|should|try to|wanna|gotta)\b",
    re.I,
)


def code_segments_planning(df, speaker_name, gap=10.0):
    refl = code_segments_reflecting(df, speaker_name, gap=gap)
    d = df.fillna("")
    turns = {t: u for t, r, w, u in _speech_turns(df, speaker_name) if r.lower() == "student" and w.strip() == (speaker_name or "").strip()}
    segs = []
    for s, e, _ in refl:
        block = [t for t in turns if s - 0.01 <= t <= e + 0.01]
        if any(_PLAN_RE.search(turns[t] or "") for t in block):
            segs.append((s, e, "PLAN_future_speech_block"))
    return _dedupe_segs(segs)


def count_all_behaviors_code(df, speaker: str):
    """Run every rule-based detector on the same raw log (modalities absent → 0 segments)."""
    return {
        "ENACTING": len(code_segments_enacting(df)),
        "MONITORING": len(code_segments_monitoring(df)),
        "INTERACTING": len(code_segments_interacting(df, speaker)),
        "REFLECTING": len(code_segments_reflecting(df, speaker)),
        "PLANNING": len(code_segments_planning(df, speaker)),
    }


_paths = [
    DAPAW_D1_MONITORING,
    DAPAW_D1_PLANNING,
    DAPAW_D1_REFLECTING,
    DAPAW_D1_ENACTING,
    DAPAW_D1_INTERACTING,
]
print("Code-only segment counts: each row is one CSV; columns are the five behaviors.\n")
for p in _paths:
    df = pd.read_csv(p).fillna("")
    cts = count_all_behaviors_code(df, _dapaw_student)
    print(p.name)
    for k in ("ENACTING", "MONITORING", "INTERACTING", "REFLECTING", "PLANNING"):
        print(f"  {k}: {cts[k]}")
    print()

print(
    "Notes: ENACTING/MONITORING use gaze/movement/state/action intersections. "
    "INTERACTING adds O2/sugar state instants + student-student speech windows. "
    "REFLECTING/PLANNING collapse the speaker's student speech; PLANNING keeps blocks with future-tense cues — "
    "this is only a coarse lower bound vs human/LLM coding."
)
